# California Housing — Data Cleaning & Exploratory Analysis

Walkthrough of the California housing dataset (20,640 districts): clean it, find what drives house prices, and visualise.

**Question:** what makes house prices higher across California?


## 1. Load the data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('housing.csv')
print(df.shape)
df.head()

## 2. Clean the data
Impute 207 missing `total_bedrooms` with the median, and flag capped values ($500,001 value, income 15, age 52).


In [ ]:
bed_median = df['total_bedrooms'].median()
df['bed_missing_flag'] = df['total_bedrooms'].isna().astype(int)
df['total_bedrooms_filled'] = df['total_bedrooms'].fillna(bed_median)
df['value_capped_flag']  = (df['median_house_value'] == 500001).astype(int)
df['income_capped_flag'] = (df['median_income'] >= 15).astype(int)
df['age_capped_flag']    = (df['housing_median_age'] == 52).astype(int)

print(f"Imputed {df['bed_missing_flag'].sum()} bedrooms (median {bed_median:g})")
print(f"Capped: {df['value_capped_flag'].sum()} value, {df['income_capped_flag'].sum()} income, {df['age_capped_flag'].sum()} age")

## 3. Summary statistics


In [ ]:
cols=['housing_median_age','total_rooms','total_bedrooms_filled','population','households','median_income','median_house_value']
df[cols].describe().T.round(2)

## 4. Segment analysis
Average house value by ocean proximity, income bracket, and latitude band.


In [ ]:
ocean=(df.groupby('ocean_proximity').agg(districts=('median_house_value','size'),
      avg_house_value=('median_house_value','mean'),avg_income=('median_income','mean'))
      .sort_values('avg_house_value',ascending=False).round(0))
ocean

In [ ]:
bins=[0,2,4,6,8,np.inf]; labels=['<2','2-4','4-6','6-8','8+']
df['income_bracket']=pd.cut(df['median_income'],bins=bins,labels=labels,right=False)
income=df.groupby('income_bracket',observed=True)['median_house_value'].mean().round(0)
income

In [ ]:
bins=[0,34,36,38,40,np.inf]
labels=['<34 (far south)','34-36 (LA/coast)','36-38 (central/Bay S)','38-40 (Bay N/Sac)','40+ (far north)']
df['lat_band']=pd.cut(df['latitude'],bins=bins,labels=labels,right=False)
lat=df.groupby('lat_band',observed=True)['median_house_value'].mean().round(0)
lat

## 5. What drives house value?


In [ ]:
feats=['median_income','total_rooms','housing_median_age','households','population']
df[feats].corrwith(df['median_house_value']).sort_values(ascending=False).round(3)

## 6. Visualise


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4))
ocean['avg_house_value'].plot(kind='bar',ax=axes[0],color='#2E75B6',title='By Ocean Proximity')
income.plot(kind='bar',ax=axes[1],color='#548235',title='By Income Bracket')
lat.plot(kind='bar',ax=axes[2],color='#C55A11',title='By Latitude Band')
for a in axes: a.set_ylabel('Avg house value ($)'); a.tick_params(axis='x',rotation=30)
plt.tight_layout(); plt.show()

## 7. Key findings
- **Median income is the dominant driver** (r ≈ 0.69); other features are weak alone.
- **Coastal premium:** coastal districts ~2× inland values (inland ≈ \$125k).
- **North–south gradient:** prices fall sharply above 38° latitude (far north ≈ \$84k vs ≈ \$220k south).
- Handle the **965 capped** and **207 imputed** rows before modelling.

**Next:** multivariate regression (income + ocean proximity + latitude), validated on a held-out sample.
